# Multi-Head Attention

Split query, key, and value vectors into heads, calculate attention in each head, then join the results. This notebook emphasizes tensor dimensions and masks.

In [ ]:
import math
import torch
from torch import nn

torch.manual_seed(29)
torch.set_num_threads(1)
print('CPU | multi-head attention shapes')

In [ ]:
batch, length, width, heads = 2, 3, 8, 2
head_width = width // heads
query = torch.randn(batch, length, width)
key = torch.randn(batch, length, width)
value = torch.randn(batch, length, width)

def split_heads(tensor):
    return tensor.view(batch, length, heads, head_width).transpose(1, 2)

q, k, v = split_heads(query), split_heads(key), split_heads(value)
scores = q @ k.transpose(-2, -1) / math.sqrt(head_width)
weights = torch.softmax(scores, dim=-1)
context = weights @ v
joined = context.transpose(1, 2).contiguous().view(batch, length, width)
assert weights.shape == (batch, heads, length, length)
assert joined.shape == (batch, length, width)
torch.testing.assert_close(weights.sum(-1), torch.ones(batch, heads, length))
print('scores:', tuple(scores.shape), 'joined heads:', tuple(joined.shape))

In [ ]:
layer = nn.MultiheadAttention(width, heads, batch_first=True)
library_output, _ = layer(query, key, value, need_weights=False)
assert library_output.shape == (batch, length, width)
print('PyTorch MultiheadAttention output:', tuple(library_output.shape))